# 📐 Structured Output Generation

**Day 3 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- How to use Gemini's native JSON mode for guaranteed structured output
- Schema enforcement with `response_schema`
- Using Pydantic models to define output shapes
- Parsing and validating structured outputs

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" pydantic

In [ ]:
import sys, os, json
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Gemini JSON Mode

Gemini can be configured to **always return valid JSON**. This is much more reliable than asking for JSON in the prompt.

In [ ]:
# JSON mode — guarantees valid JSON output
response = client.models.generate_content(
    model=MODEL_ID,
    contents="List the 5 largest planets in our solar system with their diameter in km.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

# This is guaranteed to be valid JSON
data = json.loads(response.text)
print(json.dumps(data, indent=2))

---

## 2. Schema Enforcement with `response_schema`

You can define the **exact shape** of the JSON output using a schema.

In [ ]:
# Define the schema for a book review analysis
response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Analyze this book review:
    'The Pragmatic Programmer is an essential read for any developer. 
    The examples are practical and the advice is timeless. However, 
    some chapters feel dated. Overall rating: 4.5/5.'""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "book_title": {"type": "string"},
                "sentiment": {"type": "string", "enum": ["positive", "negative", "mixed"]},
                "rating": {"type": "number"},
                "pros": {"type": "array", "items": {"type": "string"}},
                "cons": {"type": "array", "items": {"type": "string"}},
                "recommended_for": {"type": "string"},
            },
            "required": ["book_title", "sentiment", "rating", "pros", "cons"],
        },
    ),
)

result = json.loads(response.text)
print(json.dumps(result, indent=2))

---

## 3. Using Pydantic Models

**Pydantic** is a Python library for data validation. Gemini supports passing Pydantic models directly as schemas.

In [ ]:
from pydantic import BaseModel
from typing import Optional

# Define output schema using Pydantic
class JobPosting(BaseModel):
    title: str
    company: str
    location: str
    salary_min: Optional[int] = None
    salary_max: Optional[int] = None
    required_skills: list[str]
    experience_years: int
    remote_friendly: bool


response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Extract job details from this posting:
    
    Senior ML Engineer at Google, Mountain View, CA (Remote OK). 
    $180k-$250k. Requires 5+ years experience with Python, TensorFlow, 
    and distributed systems. Nice to have: Kubernetes, MLOps experience.""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=JobPosting,
    ),
)

# Parse into a Pydantic object for type-safe access
job = JobPosting.model_validate_json(response.text)
print(f"Title: {job.title}")
print(f"Company: {job.company}")
print(f"Salary: ${job.salary_min:,} - ${job.salary_max:,}")
print(f"Skills: {', '.join(job.required_skills)}")
print(f"Remote: {job.remote_friendly}")

---

## 4. Enum Outputs for Classification

Use `enum` to restrict outputs to a predefined set of values.

In [ ]:
import enum

class TicketPriority(enum.Enum):
    CRITICAL = "critical"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

class SupportTicket(BaseModel):
    category: str
    priority: TicketPriority
    summary: str
    requires_escalation: bool


tickets = [
    "The entire production database is down and no one can access the system!",
    "Can you help me change my profile picture?",
    "Our team's shared drive is running low on storage space.",
]

for ticket_text in tickets:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=f"Classify this support ticket:\n{ticket_text}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=SupportTicket,
        ),
    )
    ticket = SupportTicket.model_validate_json(response.text)
    print(f"Ticket: {ticket_text[:50]}...")
    print(f"  Priority: {ticket.priority.value} | Escalate: {ticket.requires_escalation}")
    print()

---

## 🏋️ Exercise 1: Invoice Data Extractor

Define a Pydantic model for invoice data and use Gemini to extract it.

In [ ]:
# Exercise 1: Invoice extractor
# TODO: Define a Pydantic model for an invoice

class InvoiceItem(BaseModel):
    description: str
    quantity: int
    unit_price: float
    # TODO: Add a total field

class Invoice(BaseModel):
    # TODO: Define fields for: invoice_number, date, vendor, items, subtotal, tax, total
    pass


invoice_text = """
INVOICE #INV-2024-0847
Date: March 15, 2024
From: TechSupply Corp

Items:
- 5x Wireless Mouse @ $25.00 each
- 3x USB-C Hub @ $45.00 each
- 10x HDMI Cable @ $12.00 each

Subtotal: $380.00
Tax (8%): $30.40
Total: $410.40
"""

# TODO: Use Gemini with your schema to extract the data
# response = client.models.generate_content(...)

---

## 🏋️ Exercise 2: Batch Processing with Structured Output

Process multiple customer reviews and output a summary table.

In [ ]:
# Exercise 2: Batch review processing

class ReviewAnalysis(BaseModel):
    sentiment: str  # TODO: Use enum for positive/negative/mixed
    rating: int     # 1-5
    key_topic: str

reviews = [
    "Amazing product! Fast shipping and great quality. Will buy again.",
    "Decent for the price but the build quality could be better.",
    "Terrible experience. Product arrived broken and support was unhelpful.",
    "Works as described. Nothing special but gets the job done.",
]

# TODO: Process each review with structured output and print a summary table
# Expected output: a table with columns: Review (truncated), Sentiment, Rating, Topic

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Gemini Structured Output | Docs | [ai.google.dev/gemini-api/docs/structured-output](https://ai.google.dev/gemini-api/docs/structured-output) |
| Pydantic Documentation | Docs | [docs.pydantic.dev](https://docs.pydantic.dev/) |
| JSON Mode Best Practices | Blog | [ai.google.dev/gemini-api/docs/json-mode](https://ai.google.dev/gemini-api/docs/json-mode) |

---

**Next up:** [03_ReAct_Prompting_Pattern.ipynb](./03_ReAct_Prompting_Pattern.ipynb)